# Tutorial 2: Coordinate Maps and Intensity Corrections

Before starting, make sure that a `pygid.ExpParams` instance has been successfully created based on the detector position, as shown in [Tutorial 1](tutorial_01_experimental_parameters.ipynb).


In [3]:
from pygid.datasets import get_dataset

# Download example dataset from Zenodo
try:
    files = get_dataset("tutorial_02")
    poni_path = files["poni"]
    mask_path = files["mask"]
except:
    print("Dataset download skipped on Read the Docs.")

In [4]:
import pygid
params = pygid.ExpParams(
    poni_path=poni_path,         # path to the PONI file
    ai=0.01                      # angle of incidence (degrees)
)

The `pygid.CoordMaps` class calculates and stores coordinate maps that connect detector pixel positions with reciprocal-space coordinates.

---
#### **Minimal Code Example**


In [5]:
matrix = pygid.CoordMaps(
    params,                       # pygid.ExpParams
)

---
At this step, you can also specify ranges for reciprocal vector components (`q_xy_range, q_z_range, radial_range` in Å⁻¹), angular range (`angular_range` in degrees), and resolutions (`dq` for q-values, `dang` for angles).
If not provided, `CoordMaps` will calculate the maximum available ranges based on the detector image.

To ensure consistent shapes when stacking multiple images from different experiments, it is recommended to use fixed q and angular ranges:

In [6]:
matrix = pygid.CoordMaps(
    params,
    q_xy_range=(0, 4),         # q_xy range (Å⁻¹)
    q_z_range=(0, 4),          # q_z range (Å⁻¹)
    dq=0.003,                  # q-resolution (Å⁻¹)
    radial_range=(0, 4),       # q_abs range for polar conversion (Å⁻¹)
    angular_range=(0, 90),     # angular range for polar conversion (degrees)
    dang=0.3                   # angular resolution (degrees)
)


The flags `hor_positive` and `vert_positive` can be used to include only positive values of q components:

In [ ]:
matrix = pygid.CoordMaps(
    params,
    hor_positive=False,   # only positive q_xy values (optional)
    vert_positive=False   # only positive q_z values (optional)
)

After the first conversion, coordinate maps and image axes are stored in `matrix` and can be reused. You can also **save the instance to disk** and reload it later:

In [8]:
# Save to disk
matrix = pygid.CoordMaps(
    params,
    path_to_save='matrix.pkl'  # saved as .pkl
)

# Load from disk
matrix = pygid.CoordMaps(
    path_to_load='matrix.pkl'
)


INFO - CoordMaps instance saved in matrix.pkl


## Intensity Correction

`pygid` provides a range of intensity corrections that can be applied to raw detector images.
Activation of corrections and setting of parameters is done through the `CoordMaps` instance.

### Available Corrections and Parameters

- **make_pol_corr** – polarization correction
- **pol_type** – polarization parameter (0–1):
  - `0` = vertical
  - `0.5` = unpolarized tube
  - `0.98` = synchrotron
  - `1` = horizontal
- **make_solid_angle_corr** – solid angle correction
- **make_air_attenuation_corr** – air attenuation correction
- **air_attenuation_coeff** – linear coefficient (1/m)
- **make_sensor_attenuation_corr** – sensor attenuation correction
- **sensor_attenuation_coeff** – linear coefficient (1/m)
- **sensor_thickness** – detector sensor thickness (m)
- **make_absorption_corr** – sample absorption correction
- **sample_attenuation_coeff** – linear coefficient (1/m)
- **sample_thickness** – sample thickness (m)
- **make_lorentz_corr** – Lorentz correction
- **powder_dim** – powder dimension for Lorentz correction (2 or 3)
- **dark_current** – NumPy array to subtract dark current (optional)
- **flat_field** – NumPy array for flat field normalization (optional)


In [9]:
matrix = pygid.CoordMaps(
    params,
    make_pol_corr=True,
    pol_type=0.98,
    make_solid_angle_corr=True,
    make_air_attenuation_corr=True,
    air_attenuation_coeff=1,
    make_sensor_attenuation_corr=True,
    sensor_attenuation_coeff=1,
    sensor_thickness=0.1,
    make_absorption_corr=True,
    sample_attenuation_coeff=1,
    sample_thickness=200e-9,
    make_lorentz_corr=True,
    powder_dim=3,
    dark_current=None,
    flat_field=None
)


**Note:** The actual calculations for intensity corrections are performed during the first conversion.


The next tutorial shows how detector images can be loaded and preprocessed: [Tutorial 3](tutorial_03_raw_data_loading.ipynb)